# Modern NLP Pipelines & Tokenization
**Level:** Professional | **Interactive Machine Learning Curriculum**

> Subword Tokenization (BPE, WordPiece, SentencePiece), Byte-Level BPE, Embedding spaces, and Pretraining Objectives.

---

### 📖 Conceptual Foundations

#### The Fundamental Tokenization Trade-Off

Machine learning models process numbers, not strings. Tokenization bridges this gap:
- **Word-Level Tokenization:** Suffers from massive vocabularies ($> 500,000$ words) and cannot handle Out-Of-Vocabulary (OOV) words or rare morphological variations (e.g., *unfriendliness* $\to$ `<UNK>`).
- **Character-Level Tokenization:** Zero OOV issues with a tiny vocabulary (~256 tokens), but sequences become excessively long ($5\times$ longer), crippling self-attention which scales as $O(N^2)$.
- **Subword Tokenization (BPE / WordPiece / SentencePiece):** The gold standard. Decomposes rare words into frequent subword units while keeping common words intact, achieving compact vocabularies ($32,000 - 128,000$ tokens) with zero OOV tokens.


#### Subword Tokenization Algorithms Compared

| Algorithm | Merge / Split Criterion | Whitespace Handling | Prominent Implementations |
| --- | --- | --- | --- |
| Byte-Pair Encoding (BPE) | Iteratively merges the most frequent adjacent pair of symbols in the training corpus. | Requires pre-tokenization (e.g., regex splitting on spaces). | Original BPE (Sennrich, 2016), GPT-2 / GPT-4 (Byte-level BPE). |
| WordPiece | Iteratively merges the symbol pair that maximizes the likelihood of the training corpus under a unigram language model (maximizing $\frac{\text{count}(ab)}{\text{count}(a)\text{count}(b)}$). | Uses prefix markers (e.g., `##ing`) for continuation subwords. | BERT, DistilBERT, Electra. |
| Unigram Language Model | Starts with a massive seed vocabulary and iteratively prunes the 20% of subwords whose removal increases corpus perplexity least. | Treats text as a raw stream; uses `_` to denote spaces. | SentencePiece (Kudo, 2018), T5, ALBERT, LLaMA. |
| Byte-Level BPE (BBPE) | Operates on raw UTF-8 bytes ($256$ base tokens) instead of Unicode characters. | Can encode any arbitrary character, emoji, or binary byte without `<UNK>`. | GPT-2, GPT-3, GPT-4, Llama 2/3, Mistral. |


#### BPE Merge Algorithm Walkthrough

Given a corpus of words with end-of-word markers:
`{'l o w </w>': 5, 'l o w e s t </w>': 2, 'n e w e r </w>': 6, 'w i d e r </w>': 3}`

1. Count all adjacent symbol pairs across the corpus.
2. Most frequent pair: `('e', 'r')` appearing $6 + 3 = 9$ times.
3. Merge into new token `'er'` $\to$ vocabulary gains `'er'`.
4. Repeat for $K$ merge operations until target vocabulary size is reached.


#### Pretraining Objectives: Causal vs Masked vs Span Corruption

- **Causal Language Modeling (CLM):** Next-token prediction: $\max_\theta \sum_{t=1}^T \log P(x_t \mid x_{<t}; \theta)$. Powers all modern generative autoregressive LLMs.
- **Masked Language Modeling (MLM):** Predicts randomly masked tokens ($15\%$) given both left and right context: $\max_\theta \sum_{i \in \mathcal{M}} \log P(x_i \mid \tilde{X}; \theta)$. Powers BERT.
- **Span Corruption:** Replaces random multi-token spans with sentinel tokens (e.g. `<extra_id_0>`) and decodes only the missing spans. Powers Google T5.



In [ ]:
# %matplotlib inline
import collections

def get_stats(vocab):
    pairs = collections.defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i + 1]] += freq
    return pairs

def merge_vocab(pair, v_in):
    v_out = {}
    bigram = ' '.join(pair)
    replacement = ''.join(pair)
    for word in v_in:
        # Replace adjacent pair
        w_out = word.replace(bigram, replacement)
        v_out[w_out] = v_in[word]
    return v_out

# Toy corpus
vocab = {
    'l o w </w>': 5,
    'l o w e s t </w>': 2,
    'n e w e r </w>': 6,
    'w i d e r </w>': 3,
    'n e w </w>': 2
}

print('Initial Vocabulary Counts:', vocab)
num_merges = 5

for i in range(num_merges):
    pairs = get_stats(vocab)
    if not pairs: break
    best_pair = max(pairs, key=pairs.get)
    vocab = merge_vocab(best_pair, vocab)
    print(f'Merge {i+1}: Best Pair {best_pair} (Freq: {pairs[best_pair]}) -> New Vocab sample: {list(vocab.keys())[0]}')

print('\nFinal Learned Representation:', vocab)


---

### 🎯 Hands-On Experiments & Challenges
1. **Experiment with parameters:** Modify the hyperparameters or dataset sizes in the cell above.
2. **Add visual diagnostics:** Plot loss curves or feature distributions using Matplotlib/Seaborn.
3. **Test with new data:** Try running the model on synthetic noisy data or custom inputs.


In [ ]:
# Your custom experiment code here:

